# Predict this week

Set `SEASON` and `WEEK`, then run all cells. The table is the predicted winner, win probability, and the biggest reasons (style matchup, weather, injuries, form).

First run downloads nflverse data and can take several minutes. Later runs use the local cache.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from nfl_predictor.config import current_nfl_season
from nfl_predictor.models.predict import default_week, predict_week
from nfl_predictor.models.train import ensure_model
from nfl_predictor.pipeline import build_feature_table

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

SEASON = current_nfl_season()
WEEK = None  # None = next unplayed week
SEASON, WEEK

(2026, None)

In [2]:
features = build_feature_table()
if WEEK is None:
    WEEK = default_week(features, SEASON)
print(f"Predicting {SEASON} week {WEEK}")
artifact = ensure_model(features)
preds = predict_week(features, season=SEASON, week=WEEK, artifact=artifact)
preds

Loading schedules 2018-2026...
Loading play-by-play team-game stats (first run is slow)...
Loading injuries and snap counts...
Filling weather for upcoming outdoor games...
Building pre-kickoff features...
Predicting 2026 week 2


,season,week,gameday,away_team,home_team,home_style,away_style,home_injury_penalty,away_injury_penalty,temp,...,is_snow,is_heat,is_indoor,home_win_prob,away_win_prob,predicted_winner,predicted_win_prob,reasons,home_score,away_score
0,2026,2,2026-09-17,DET,BUF,run-heavy,balanced,0.076,0.206,70.500,...,0.000,0.000,0.000,0.730,0.270,BUF,0.730,Home offense EPA/play helps home; Home pass EPA helps away; Away EPA allowed vs rush helps home; Home EPA allowed vs...,NaN,NaN
1,2026,2,2026-09-20,SEA,ARI,pass-heavy,balanced,0.000,0.000,98.200,...,0.000,1.000,0.000,0.371,0.629,SEA,0.629,Home blended form point diff helps away; Away team road point diff helps home; Away points allowed helps away; Home ...,NaN,NaN
2,2026,2,2026-09-20,CAR,ATL,run-heavy,balanced,0.000,0.000,87.700,...,0.000,1.000,0.000,0.573,0.427,ATL,0.573,Home offense EPA/play helps away; Home pass EPA helps home; Home injury penalty helps home; Away games in sample hel...,NaN,NaN
3,2026,2,2026-09-20,NO,BAL,run-heavy,balanced,0.000,0.000,83.300,...,0.000,0.000,0.000,0.682,0.318,BAL,0.682,Home offense EPA/play helps home; Home rush EPA helps away; Home pass EPA helps away; Home injury penalty helps home...,NaN,NaN
4,2026,2,2026-09-20,MIN,CHI,balanced,run-heavy,0.000,0.000,64.600,...,0.000,0.000,0.000,0.570,0.430,CHI,0.570,Home offense EPA/play helps home; Home pass EPA helps away; Away points allowed helps away; Home rush EPA helps away...,NaN,NaN
5,2026,2,2026-09-20,WAS,DAL,balanced,run-heavy,0.000,0.000,99.800,...,0.000,1.000,0.000,0.604,0.396,DAL,0.604,Home offense EPA/play helps home; Home EPA allowed vs rush helps away; Home blended form point diff helps away; Home...,NaN,NaN
6,2026,2,2026-09-20,JAX,DEN,balanced,balanced,0.000,0.000,78.800,...,0.000,0.000,0.000,0.253,0.747,JAX,0.747,Away points per game helps away; Away pass EPA helps away; Home offense EPA/play helps away; Home injury penalty hel...,NaN,NaN
7,2026,2,2026-09-20,CIN,HOU,run-heavy,balanced,0.000,0.000,93.900,...,0.000,1.000,0.000,0.555,0.445,HOU,0.555,Away points per game helps away; Home injury penalty helps home; Away games in sample helps away; Home games in samp...,NaN,NaN
8,2026,2,2026-09-20,IND,KC,balanced,balanced,0.000,0.000,72.700,...,0.000,0.000,0.000,0.591,0.409,KC,0.591,Home offense EPA/play helps away; Home pass EPA helps home; Away pass EPA helps home; Away points allowed helps home...,NaN,NaN
9,2026,2,2026-09-20,LV,LAC,balanced,balanced,0.000,0.000,NaN,...,0.000,0.000,1.000,0.622,0.378,LAC,0.622,Home offense EPA/play helps away; Away points per game helps home; Away pass EPA helps home; Home pass EPA helps hom...,NaN,NaN


In [3]:
if preds.empty:
    available = (
        features.loc[features["season"] == SEASON, "week"]
        .dropna()
        .astype(int)
        .sort_values()
        .unique()
        .tolist()
    )
    print(f"No games found for {SEASON} week {WEEK}. Available weeks: {available}")
else:
    view = preds.copy()
    view["matchup"] = view["away_team"] + " @ " + view["home_team"]
    view["weather"] = view.apply(
        lambda r: "indoor" if r.get("is_indoor") else f"{r.get('temp')} F, wind {r.get('wind')}",
        axis=1,
    )
    show = view[
        [
            "gameday",
            "matchup",
            "away_style",
            "home_style",
            "away_injury_penalty",
            "home_injury_penalty",
            "weather",
            "predicted_winner",
            "predicted_win_prob",
            "reasons",
        ]
    ]
    display(show)

,gameday,matchup,away_style,home_style,away_injury_penalty,home_injury_penalty,weather,predicted_winner,predicted_win_prob,reasons
0,2026-09-17,DET @ BUF,balanced,run-heavy,0.206,0.076,"70.5 F, wind 3.4",BUF,0.730,Home offense EPA/play helps home; Home pass EPA helps away; Away EPA allowed vs rush helps home; Home EPA allowed vs...
1,2026-09-20,SEA @ ARI,balanced,pass-heavy,0.000,0.000,"98.2 F, wind 9.4",SEA,0.629,Home blended form point diff helps away; Away team road point diff helps home; Away points allowed helps away; Home ...
2,2026-09-20,CAR @ ATL,balanced,run-heavy,0.000,0.000,"87.7 F, wind 3.7",ATL,0.573,Home offense EPA/play helps away; Home pass EPA helps home; Home injury penalty helps home; Away games in sample hel...
3,2026-09-20,NO @ BAL,balanced,run-heavy,0.000,0.000,"83.3 F, wind 11.7",BAL,0.682,Home offense EPA/play helps home; Home rush EPA helps away; Home pass EPA helps away; Home injury penalty helps home...
4,2026-09-20,MIN @ CHI,run-heavy,balanced,0.000,0.000,"64.6 F, wind 13.0",CHI,0.570,Home offense EPA/play helps home; Home pass EPA helps away; Away points allowed helps away; Home rush EPA helps away...
5,2026-09-20,WAS @ DAL,run-heavy,balanced,0.000,0.000,"99.8 F, wind 9.0",DAL,0.604,Home offense EPA/play helps home; Home EPA allowed vs rush helps away; Home blended form point diff helps away; Home...
6,2026-09-20,JAX @ DEN,balanced,balanced,0.000,0.000,"78.8 F, wind 7.8",JAX,0.747,Away points per game helps away; Away pass EPA helps away; Home offense EPA/play helps away; Home injury penalty hel...
7,2026-09-20,CIN @ HOU,balanced,run-heavy,0.000,0.000,"93.9 F, wind 6.2",HOU,0.555,Away points per game helps away; Home injury penalty helps home; Away games in sample helps away; Home games in samp...
8,2026-09-20,IND @ KC,balanced,balanced,0.000,0.000,"72.7 F, wind 10.6",KC,0.591,Home offense EPA/play helps away; Home pass EPA helps home; Away pass EPA helps home; Away points allowed helps home...
9,2026-09-20,LV @ LAC,balanced,balanced,0.000,0.000,indoor,LAC,0.622,Home offense EPA/play helps away; Away points per game helps home; Away pass EPA helps home; Home pass EPA helps hom...


Probabilities are P(that team wins the game). They will not sum to anything across the slate; each game is independent. If a game already has a final score, you can compare `predicted_winner` to the actual result in the first table.